# OSL DRQN/DQN Notebook (Legacy demo)

Discrete-action recurrent / feed-forward Q-learning baseline for the **legacy** single-sensor 2D OSL env (Gaussian / turbulent plume + 4-step cast macro). Lives in `demo/DRQN/` — kept as a historical reference before the biological refactor (bilateral sensor + larva connectome PPO/RSAC under `src/`).

- Env: legacy `StaticEnv` / `DynamicEnv` from `demo/DRQN/osl_env_2d.py` — obs `[c, did_cast]`, action `Box([v, omega, cast])` with 4-step cast lock.
- Agent: `DRQNAgent` (recurrent=True for DRQN, False for vanilla DQN) over discrete action set `{RUN, CAST, TURN_L, TURN_R}` mapped to the env's continuous action via `to_env_action`.

Repo clone is required (the demo module imports as `demo.DRQN.*`).

In [ ]:
# Colab setup: clone repo + install deps + cd in. Re-running is safe.

import os, sys, subprocess

REPO_URL = 'https://github.com/InHyunseo/Brain-inspired-OSL.git'
REPO_DIR = '/content/2d-osl' if os.path.isdir('/content') else os.path.abspath('2d-osl')

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%pip install --quiet -r requirements.txt scipy

print('repo:', REPO_DIR)
print('cwd :', os.getcwd())

In [ ]:
import os, io
from collections import deque

import numpy as np
import torch
import matplotlib.pyplot as plt
import imageio
from IPython.display import Image as DisplayImage, display

from demo.DRQN import (
    StaticEnv, DynamicEnv,
    DRQNAgent,
    A_RUN, A_CAST, A_TURN_L, A_TURN_R, N_ACTIONS,
    EpisodeReplayBuffer,
)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

os.makedirs('checkpoints_drqn', exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 1. Train

4-phase curriculum (`static → dynamic_0.3 → 0.6 → 1.0`). DRQN is single-env (no SubprocVecEnv), so episode counts are scaled down vs PPO. Toggle `RECURRENT = False` for vanilla DQN.

### Live TensorBoard

In [ ]:
# ===== TensorBoard (run BEFORE the training cell to watch curves live) =====
# Writer location in DRQN cell: checkpoints_drqn/tb_<timestamp>/
import os
os.makedirs('checkpoints_drqn', exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir checkpoints_drqn --port 6006


In [ ]:
import os, time
# ===== HYPERPARAMETERS — edit freely =====
RECURRENT = True   # True = DRQN (GRU), False = vanilla DQN (MLP)
PHASES = [
    ('static',      20_000),
    ('dynamic_0.3', 10_000),
    ('dynamic_0.6', 10_000),
    ('dynamic_1.0', 20_000),
]

AGENT_KW = dict(rnn_hidden=147, lr=1e-4, gamma=0.99)
BATCH_SIZE = 128
SEQ_LEN = 16
BUFFER_CAP_STEPS = 150_000
LEARNING_STARTS = 5_000
TARGET_SYNC_EVERY = 20  # episodes

EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY_EPS = 4_000

LOG_EVERY = 200
SEED = 42
# ==========================================

def make_env(env_kind):
    if env_kind == 'static':
        return StaticEnv()
    return DynamicEnv(noise_coef=float(env_kind.split('_')[1]))

def linear_eps(ep):
    frac = min(1.0, ep / max(1, EPS_DECAY_EPS))
    return EPS_START + frac * (EPS_END - EPS_START)

# TensorBoard writer (mirrors per-LOG_EVERY scalars)
try:
    from torch.utils.tensorboard import SummaryWriter
    TB_DIR = os.path.join('checkpoints_drqn', f'tb_{time.strftime("%Y%m%d_%H%M%S")}')
    os.makedirs(TB_DIR, exist_ok=True)
    tb_writer = SummaryWriter(log_dir=TB_DIR)
except Exception as exc:
    print(f'[tb] disabled ({exc})')
    tb_writer = None

env = make_env(PHASES[0][0])
agent = DRQNAgent(
    obs_dim=env.observation_space.shape[0],
    action_low=env.action_space.low, action_high=env.action_space.high,
    device=DEVICE, recurrent=RECURRENT, **AGENT_KW,
)
buffer = EpisodeReplayBuffer(cap_steps=BUFFER_CAP_STEPS)

ep_returns, success_window = [], deque(maxlen=100)
global_ep = 0

for phase_kind, phase_episodes in PHASES:
    env = make_env(phase_kind)
    print(f'=== Phase {phase_kind} ({phase_episodes} eps) ===')
    for _ in range(phase_episodes):
        global_ep += 1
        obs, _ = env.reset(seed=SEED + global_ep)
        h, traj, ep_ret, success = None, [], 0.0, False
        while True:
            eps = linear_eps(global_ep)
            a_idx, h = agent.get_action(obs, h, epsilon=eps)
            next_obs, r, term, trunc, info = env.step(agent.to_env_action(a_idx))
            traj.append((obs, a_idx, float(r), next_obs, float(term)))
            obs = next_obs
            ep_ret += float(r)
            if info.get('is_success'):
                success = True
            if term or trunc:
                break
        buffer.add_episode(traj)
        ep_returns.append(ep_ret)
        success_window.append(float(success))

        if len(buffer) >= LEARNING_STARTS:
            agent.update(buffer.sample(batch_size=BATCH_SIZE, seq_len=SEQ_LEN))
            if global_ep % TARGET_SYNC_EVERY == 0:
                agent.sync_target()

        if global_ep % LOG_EVERY == 0:
            print(f'  ep {global_ep:5d}  ret={ep_ret:7.2f}  succ100={np.mean(success_window):.3f}  eps={eps:.3f}')
            if tb_writer is not None:
                tb_writer.add_scalar('return/episode', float(ep_ret), global_ep)
                tb_writer.add_scalar('return/recent_mean', float(np.mean(ep_returns[-LOG_EVERY:])), global_ep)
                tb_writer.add_scalar('success/window100', float(np.mean(success_window)), global_ep)
                tb_writer.add_scalar('epsilon', float(eps), global_ep)
                tb_writer.add_scalar('buffer/size', int(len(buffer)), global_ep)
                tb_writer.flush()

    agent.save(f'checkpoints_drqn/{phase_kind}.pt')
    print(f'  saved checkpoints_drqn/{phase_kind}.pt')

if tb_writer is not None:
    tb_writer.close()


## 2. Training-curve plot

In [ ]:
def ema(arr, alpha=0.05):
    out, m = [], arr[0] if arr else 0.0
    for x in arr:
        m = alpha * x + (1 - alpha) * m
        out.append(m)
    return out

fig, ax = plt.subplots(figsize=(10, 4))
x = list(range(1, len(ep_returns) + 1))
ax.plot(x, ep_returns, alpha=0.25, label='raw')
ax.plot(x, ema(ep_returns), linewidth=2, label='ema')
ax.set_xlabel('episode'); ax.set_ylabel('return'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 3. Evaluate + Elite Seed Search

Find seeds where the deterministic policy succeeds with a sensible cast count, then render the best one as an inline GIF (white star markers = cast events).

In [ ]:
# ===== EVAL HYPERPARAMETERS =====
EVAL_ENV_KIND = 'dynamic_1.0'
EVAL_N_TO_FIND = 3
EVAL_TRIALS = 500
EVAL_SEED_BASE = 20_000
MIN_CASTS = 1
MAX_CASTS = 300
GIF_FILENAME = 'drqn_turbulent.gif'
# ================================

def find_elite_seeds(agent, env_kind=EVAL_ENV_KIND, n_to_find=EVAL_N_TO_FIND,
                     min_casts=MIN_CASTS, max_casts=MAX_CASTS):
    env = make_env(env_kind)
    elites = []
    for trial in range(EVAL_TRIALS):
        seed = EVAL_SEED_BASE + trial
        obs, _ = env.reset(seed=seed)
        h = None; casts = 0; success = False
        for _ in range(300):
            a_idx, h = agent.get_action_deterministic(obs, h)
            if a_idx == A_CAST:
                casts += 1
            obs, _, term, trunc, info = env.step(agent.to_env_action(a_idx))
            if info.get('is_success'):
                success = True
            if term or trunc:
                break
        if success and min_casts <= casts <= max_casts:
            elites.append(seed)
            print(f'[seed {seed}] success, casts={casts}')
        if len(elites) >= n_to_find:
            break
    return elites

def render_elite_gif(agent, seed, env_kind=EVAL_ENV_KIND, filename=GIF_FILENAME):
    env = make_env(env_kind)
    obs, _ = env.reset(seed=seed)
    h = None
    frames, traj_x, traj_y, cx, cy = [], [], [], [], []
    for t in range(300):
        a_idx, h = agent.get_action_deterministic(obs, h)
        traj_x.append(env.x); traj_y.append(env.y)
        if a_idx == A_CAST or getattr(env, 'in_cast', False):
            cx.append(env.x); cy.append(env.y)

        field = getattr(env, '_field_view', None)
        if field is None:
            res = 100
            xs = np.linspace(-env.L, env.L, res); ys = np.linspace(-env.L, env.L, res)
            X, Y = np.meshgrid(xs, ys)
            field = np.clip(np.asarray(env._conc(X, Y), dtype=np.float32), 0.0, 1.0)

        fig, ax = plt.subplots(figsize=(7, 7))
        fig.patch.set_facecolor('black'); ax.set_facecolor('black')
        ax.imshow(field, extent=[-env.L, env.L, -env.L, env.L], origin='lower', cmap='inferno', vmin=0, vmax=1)
        ax.plot(traj_x, traj_y, color='#50dcff', linewidth=2.0, alpha=0.8)
        if cx:
            ax.scatter(cx, cy, color='white', marker='*', s=150, edgecolors='black', zorder=10)
        ax.scatter(env.src_x, env.src_y, color='lime', marker='P', s=150, zorder=11)
        ax.add_patch(plt.Circle((env.src_x, env.src_y), env.r_goal, color='gray', fill=False))
        ax.set_title(f'DRQN {env_kind} seed={seed} step={t}', color='white')
        ax.tick_params(colors='white')

        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=80, bbox_inches='tight', facecolor='black')
        plt.close(fig); buf.seek(0)
        frames.append(np.array(imageio.v2.imread(buf)))

        obs, _, term, trunc, _ = env.step(agent.to_env_action(a_idx))
        if term or trunc:
            break

    imageio.mimsave(filename, frames, fps=15)
    display(DisplayImage(data=open(filename, 'rb').read(), format='gif'))

elites = find_elite_seeds(agent)
if elites:
    render_elite_gif(agent, elites[0])
else:
    print('no elite seed found')